In [0]:
#| default_exp skill

## Installing the skill

Install the notebook workflow instructions for agents.

In [0]:
#| export
from importlib.resources import files
from pathlib import Path

from fastcore.script import call_parse

from nbskill.foundation import _cli_return, _tracked_call

In [ ]:
#| export
@call_parse
@_tracked_call
def install_nbskill(
    target: str = "codex",  # codex, claude, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Overwrite an existing SKILL.md
):
    "Install the bundled SKILL.md into a Codex or Claude Code skills directory."
    target = target.lower()
    if skills_dir: roots = [Path(skills_dir).expanduser()]
    elif target == "codex": roots = [Path.home() / ".codex" / "skills"]
    elif target in {"claude", "claude-code", "claude_code"}: roots = [Path.home() / ".claude" / "skills"]
    elif target == "both": roots = [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    else: raise ValueError("target must be codex, claude, both, or use skills_dir")
    package = files("nbskill")
    skill_text = package.joinpath("SKILL.md").read_text(encoding="utf-8")
    references = package.joinpath("references")
    installed = []
    for root in roots:
        dst_dir = root / skill_name
        dst = dst_dir / "SKILL.md"
        if dst.exists() and not overwrite: raise FileExistsError(dst)
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst.write_text(skill_text, encoding="utf-8")
        if references.is_dir():
            ref_dir = dst_dir / "references"
            ref_dir.mkdir(exist_ok=True)
            for ref in references.iterdir():
                if ref.is_file():
                    (ref_dir / ref.name).write_text(ref.read_text(encoding="utf-8"), encoding="utf-8")
        installed.append(dst)
    msg = "\n".join(f"Installed {path}" for path in installed)
    print(msg)
    return _cli_return(installed)

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path
from nbskill.skill import install_nbskill

with _tempfile.TemporaryDirectory() as td:
    install_nbskill(skills_dir=td)
    skill_dir = _Path(td) / "jupyter-notebooks"
    assert (skill_dir / "SKILL.md").exists()
    assert (skill_dir / "references" / "mcp-tools.md").exists()
    assert (skill_dir / "references" / "cli-fallbacks.md").exists()